# WiSARD Dataset Exploration

Understanding the structure, quality, and diversity of the full WiSARD Multi-Modal dataset for paired RGB-thermal SSL training.

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["font.size"] = 10

# Find the project root by looking for data/processed/wisard-full
notebook_dir = Path(".").resolve()
possible_roots = [
    notebook_dir,  # Current directory
    notebook_dir.parent,  # Parent (reports is in root)
    Path("/Users/eoinmcallister/Projects/ssl-aerial-person-detection"),
]

PROCESSED_ROOT = None
for root in possible_roots:
    candidate = root / "data" / "processed" / "wisard-full"
    if candidate.exists():
        PROCESSED_ROOT = candidate
        break

if PROCESSED_ROOT is None:
    raise FileNotFoundError("Cannot find data/processed/wisard-full directory")


def load_records(filename):
    path = PROCESSED_ROOT / filename
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines()]


train = load_records("train.jsonl")
val = load_records("validation.jsonl")
test = load_records("test.jsonl")

print(f"Dataset:")
print(f"  Train:      {len(train):,} pairs")
print(f"  Validation: {len(val):,} pairs")
print(f"  Test:       {len(test):,} pairs")
print(f"  Total:      {len(train) + len(val) + len(test):,} pairs")

## Box Count Distributions

How many people (boxes) does each modality typically see per image?

In [ ]:
def extract_box_counts(records):
    rgb_counts = [len(r.get("rgb_boxes", [])) for r in records]
    thermal_counts = [len(r.get("thermal_boxes", [])) for r in records]
    return rgb_counts, thermal_counts


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, records) in zip(
    axes, [("Train", train), ("Validation", val), ("Test", test)]
):
    if not records:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        continue

    rgb, thermal = extract_box_counts(records)

    ax.hist(
        rgb,
        bins=range(0, max(rgb) + 2),
        alpha=0.6,
        label="RGB",
        color="tab:blue",
        edgecolor="black",
    )
    ax.hist(
        thermal,
        bins=range(0, max(thermal) + 2),
        alpha=0.6,
        label="Thermal",
        color="tab:orange",
        edgecolor="black",
    )
    ax.set_xlabel("Boxes per image")
    ax.set_ylabel("Frequency")
    ax.set_title(f"{name} (n={len(records)})")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Annotation Density Summary

In [ ]:
def stats_per_split(records, name):
    if not records:
        return {}
    rgb, thermal = extract_box_counts(records)
    return {
        "Split": name,
        "Pairs": len(records),
        "RGB mean": f"{np.mean(rgb):.2f}",
        "Thermal mean": f"{np.mean(thermal):.2f}",
        "RGB total": sum(rgb),
        "Thermal total": sum(thermal),
        "Empty RGB": sum(1 for c in rgb if c == 0),
        "Empty Thermal": sum(1 for c in thermal if c == 0),
    }


stats_list = [
    stats_per_split(train, "Train"),
    stats_per_split(val, "Validation"),
    stats_per_split(test, "Test"),
]
stats_df = pd.DataFrame([s for s in stats_list if s])
print(stats_df.to_string(index=False))

## RGB vs Thermal Box Count Comparison

Scatter plot: each point is one image pair. Points on the diagonal = perfect agreement.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, records) in zip(
    axes, [("Train", train), ("Validation", val), ("Test", test)]
):
    if not records:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        continue

    rgb, thermal = extract_box_counts(records)

    # Add jitter to see overlaps
    rgb_jitter = np.array(rgb) + np.random.normal(0, 0.05, len(rgb))
    thermal_jitter = np.array(thermal) + np.random.normal(0, 0.05, len(thermal))

    ax.scatter(rgb_jitter, thermal_jitter, alpha=0.4, s=20, color="tab:purple")

    # Diagonal line (perfect agreement)
    max_val = max(max(rgb), max(thermal))
    ax.plot(
        [0, max_val],
        [0, max_val],
        "r--",
        alpha=0.5,
        linewidth=2,
        label="Perfect agreement",
    )

    ax.set_xlabel("RGB boxes")
    ax.set_ylabel("Thermal boxes")
    ax.set_title(f"{name} (n={len(records)})")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Agreement Patterns

Which images have matching box counts (agreement) vs mismatches (complementarity)?

In [ ]:
def agreement_breakdown(records, name):
    if not records:
        return None

    rgb, thermal = extract_box_counts(records)

    # Categorize
    agree = sum(1 for r, t in zip(rgb, thermal) if r == t)
    rgb_more = sum(1 for r, t in zip(rgb, thermal) if r > t)
    thermal_more = sum(1 for r, t in zip(rgb, thermal) if t > r)

    total = len(records)

    return {
        "Split": name,
        "Both empty": sum(1 for r, t in zip(rgb, thermal) if r == 0 and t == 0),
        "Agree (boxes match)": agree,
        "RGB sees more": rgb_more,
        "Thermal sees more": thermal_more,
        "Agreement %": f"{100 * agree / total:.1f}%",
    }


agree_list = [
    agreement_breakdown(train, "Train"),
    agreement_breakdown(val, "Validation"),
    agreement_breakdown(test, "Test"),
]
agree_df = pd.DataFrame([a for a in agree_list if a])
print(agree_df.to_string(index=False))
print()
print(
    "Note: Disagreement (RGB/Thermal seeing different counts) is VALUABLE for SSL training."
)
print("It means the modalities are learning complementary features.")

## Modality Complementarity

Cases where one modality detects people the other misses (no boxes vs has boxes).

In [ ]:
def complementarity_analysis(records, name):
    if not records:
        return None

    rgb, thermal = extract_box_counts(records)

    # Cases where one is empty and the other is not
    rgb_empty_thermal_not = sum(1 for r, t in zip(rgb, thermal) if r == 0 and t > 0)
    thermal_empty_rgb_not = sum(1 for r, t in zip(rgb, thermal) if t == 0 and r > 0)
    neither_empty = sum(1 for r, t in zip(rgb, thermal) if r > 0 and t > 0)
    both_empty = sum(1 for r, t in zip(rgb, thermal) if r == 0 and t == 0)

    total = len(records)

    return {
        "Split": name,
        "Total pairs": total,
        "Both see people": neither_empty,
        "RGB only": thermal_empty_rgb_not,
        "Thermal only": rgb_empty_thermal_not,
        "Neither sees people": both_empty,
        "Complementarity %": f"{100 * (rgb_empty_thermal_not + thermal_empty_rgb_not) / total:.1f}%",
    }


comp_list = [
    complementarity_analysis(train, "Train"),
    complementarity_analysis(val, "Validation"),
    complementarity_analysis(test, "Test"),
]
comp_df = pd.DataFrame([c for c in comp_list if c])
print(comp_df.to_string(index=False))
print()
print("Complementarity: Cases where RGB and thermal see different PRESENCE of people.")
print("This is exactly what SSL needs to learn meaningful shared representations.")

## Sample Pairs: Visual Inspection

Representative examples of agreement, complementarity, and disagreement in action.

In [ ]:
from PIL import Image
import matplotlib.patches as patches

DATA_ROOT = Path("data/raw/wisard-full")


# Find representative cases from each split
def find_cases(split_records):
    if not split_records:
        return {}

    rgb_only = None
    thermal_only = None
    both_agree = None
    high_disagree = None

    for i, record in enumerate(split_records):
        rgb_count = len(record.get("rgb_boxes", []))
        thermal_count = len(record.get("thermal_boxes", []))

        if rgb_only is None and rgb_count > 0 and thermal_count == 0:
            rgb_only = i
        if thermal_only is None and thermal_count > 0 and rgb_count == 0:
            thermal_only = i
        if both_agree is None and rgb_count == thermal_count and rgb_count > 0:
            both_agree = i
        if (
            high_disagree is None
            and rgb_count > 0
            and thermal_count > 0
            and rgb_count != thermal_count
        ):
            high_disagree = i

        if all([rgb_only, thermal_only, both_agree, high_disagree]):
            break

    return {
        "rgb_only": rgb_only,
        "thermal_only": thermal_only,
        "both_agree": both_agree,
        "high_disagree": high_disagree,
    }


cases = find_cases(train)

fig, axes = plt.subplots(4, 2, figsize=(12, 14))

case_names = ["rgb_only", "thermal_only", "both_agree", "high_disagree"]
case_labels = [
    "RGB sees people, thermal empty",
    "Thermal sees people, RGB empty",
    "Both agree on count",
    "Disagreement: different counts",
]

for row, (case_name, label) in enumerate(zip(case_names, case_labels)):
    if cases[case_name] is None:
        axes[row, 0].text(
            0.5,
            0.5,
            "No example found",
            ha="center",
            va="center",
            transform=axes[row, 0].transAxes,
        )
        axes[row, 1].text(
            0.5,
            0.5,
            "No example found",
            ha="center",
            va="center",
            transform=axes[row, 1].transAxes,
        )
        continue

    record = train[cases[case_name]]
    rgb_count = len(record["rgb_boxes"])
    thermal_count = len(record["thermal_boxes"])

    # RGB image
    rgb_path = DATA_ROOT / record["rgb_image"]
    if rgb_path.exists():
        rgb_img = Image.open(rgb_path)
        axes[row, 0].imshow(rgb_img)
        axes[row, 0].set_title(f"{label}\nRGB: {rgb_count} boxes")
        # Draw boxes
        for box in record["rgb_boxes"]:
            h, w = rgb_img.height, rgb_img.width
            x_min = (box["x_center"] - box["width"] / 2) * w
            y_min = (box["y_center"] - box["height"] / 2) * h
            x_max = (box["x_center"] + box["width"] / 2) * w
            y_max = (box["y_center"] + box["height"] / 2) * h
            rect = patches.Rectangle(
                (x_min, y_min),
                x_max - x_min,
                y_max - y_min,
                linewidth=2,
                edgecolor="g",
                facecolor="none",
            )
            axes[row, 0].add_patch(rect)
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])

    # Thermal image
    thermal_path = DATA_ROOT / record["thermal_image"]
    if thermal_path.exists():
        thermal_img = Image.open(thermal_path)
        axes[row, 1].imshow(thermal_img, cmap="hot")
        axes[row, 1].set_title(f"Thermal: {thermal_count} boxes")
        # Draw boxes
        for box in record["thermal_boxes"]:
            h, w = thermal_img.height, thermal_img.width
            x_min = (box["x_center"] - box["width"] / 2) * w
            y_min = (box["y_center"] - box["height"] / 2) * h
            x_max = (box["x_center"] + box["width"] / 2) * w
            y_max = (box["y_center"] + box["height"] / 2) * h
            rect = patches.Rectangle(
                (x_min, y_min),
                x_max - x_min,
                y_max - y_min,
                linewidth=2,
                edgecolor="r",
                facecolor="none",
            )
            axes[row, 1].add_patch(rect)
    axes[row, 1].set_xticks([])
    axes[row, 1].set_yticks([])

plt.tight_layout()
plt.show()

## Collection and Flight Diversity

How many flights are in each split, and how are pairs distributed?

In [ ]:
# Group by collection and analyze diversity
from collections import Counter

splits_with_collections = {
    "Train": train,
    "Validation": val,
    "Test": test,
}

diversity_data = []
for split_name, records in splits_with_collections.items():
    if not records:
        continue

    collections = [r["collection_id"] for r in records]
    collection_counts = Counter(collections)
    unique_collections = len(collection_counts)

    diversity_data.append(
        {
            "Split": split_name,
            "Unique flights": unique_collections,
            "Total pairs": len(records),
            "Pairs per flight (mean)": f"{len(records) / unique_collections:.1f}",
            "Top flight %": f"{100 * max(collection_counts.values()) / len(records):.1f}%",
        }
    )

diversity_df = pd.DataFrame(diversity_data)
print("Flight Distribution Across Splits:")
print(diversity_df.to_string(index=False))

# Bar chart: number of flights per split
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

for split_name, records in splits_with_collections.items():
    if not records:
        continue
    collections = [r["collection_id"] for r in records]
    collection_counts = Counter(collections)
    ax1.bar(split_name, len(collection_counts), color="tab:blue", alpha=0.7)

ax1.set_ylabel("Number of unique flights")
ax1.set_title("Flights per split (collection-level splits ensure no flight is split)")
ax1.grid(True, alpha=0.3, axis="y")

# Pairs per flight distribution
for split_name, records in splits_with_collections.items():
    if not records:
        continue
    collections = [r["collection_id"] for r in records]
    collection_counts = Counter(collections)
    pairs_per_flight = list(collection_counts.values())
    ax2.hist(pairs_per_flight, bins=20, alpha=0.5, label=split_name, edgecolor="black")

ax2.set_xlabel("Pairs per flight")
ax2.set_ylabel("Frequency")
ax2.set_title("Distribution of pairs per flight")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Data Quality Transparency Log

Pragmatic pairing decisions: what we accepted and why.

In [ ]:
import json

quality_path = PROCESSED_ROOT / "data_quality.json"
if quality_path.exists():
    with open(quality_path) as f:
        quality = json.load(f)

    print("Data Quality Log - Pragmatic Pairing Decisions")
    print("=" * 70)
    print()
    print(f"Frames skipped (missing annotations): {quality.get('frames_skipped', 0):,}")
    print("  Why: Real datasets have partial labeling. Unlabeled frames are")
    print("       not errors; they simply aren't used.")
    print()
    print(
        f"Boxes clamped (out-of-range coordinates): {quality.get('boxes_clamped', 0)}"
    )
    print("  Why: Rare annotation errors (e.g., boxes slightly outside [0,1]).")
    print("       Clamping fixes them without losing the box.")
    print()
    print(
        f"Flights with multi-variant pairing: {quality.get('flights_multi_variant', 0)}"
    )
    print("  Why: Some locations have multiple VIS/IR recordings (different times,")
    print("       angles, or sensors). We pair them all, using min(VIS, IR) variants")
    print("       per location, rather than dropping multi-variant flights.")
    print()
    print("=" * 70)
    print()
    print("Outcome: 7,359 pairs extracted with real-world messiness accepted,")
    print("         not forced away.")
else:
    print("data_quality.json not found")

## Label-Fraction Sanity Check

Before Stage 2 (SSL sweep), verify that proposed label percentages are viable.

In [ ]:
if train:
    rgb, thermal = extract_box_counts(train)
    total_rgb = sum(rgb)
    total_thermal = sum(thermal)
    total_boxes = total_rgb + total_thermal

    fractions = [0.01, 0.05, 0.10, 1.0]
    label_data = []

    for frac in fractions:
        num_pairs = int(len(train) * frac)
        num_rgb_boxes = int(total_rgb * frac)
        num_thermal_boxes = int(total_thermal * frac)
        num_total_boxes = num_rgb_boxes + num_thermal_boxes

        label_data.append(
            {
                "Label %": f"{frac * 100:.0f}%",
                "Pairs": num_pairs,
                "RGB boxes": num_rgb_boxes,
                "Thermal boxes": num_thermal_boxes,
                "Total boxes": num_total_boxes,
            }
        )

    label_df = pd.DataFrame(label_data)
    print("\nLabel-Fraction Breakdown (from train split):")
    print(label_df.to_string(index=False))
    print()
    print("Interpretation:")
    print("- 1%:  ~59 pairs, ~131 total boxes — small but usable")
    print("- 5%:  ~293 pairs, ~655 total boxes — moderate labeling effort")
    print("- 10%: ~587 pairs, ~1,310 total boxes — substantial but affordable")
    print("- 100%: ~5,867 pairs, ~13,098 total boxes — full dataset")
    print()
    print("All fractions are viable. 1% represents real constraint (59 pairs is a")
    print("few hours of manual labeling); 10% is realistic for a single annotator.")
else:
    print("No train records loaded")

## Summary

**Dataset is ready for SSL training:**

- ✓ Large, real SAR data with diverse conditions
- ✓ Dense annotations (2+ boxes/image average)
- ✓ Realistic disagreement between modalities (30-57% depending on split)
- ✓ Significant complementarity (modalities catch different people in 15-25% of cases)
- ✓ No flight leakage (collection-level splits)

**Next: Train contrastive encoder on paired RGB-thermal data.**